# LEAA Training — Stage 5: `wind`
**Target accuracy:** 55%  |  **Timesteps:** 20,000,000

### Setup Instructions
1. **Runtime → Change runtime type → T4 GPU**
2. Add secrets in the left sidebar (🔑 Secrets):
   - `GITHUB_TOKEN` — your GitHub Personal Access Token
   - `GMAIL_ADDRESS` — your Gmail address *(optional, for notifications)*
   - `GMAIL_APP_PASSWORD` — Gmail App Password *(optional)*
     → [Create App Password](https://myaccount.google.com/apppasswords) (requires 2FA enabled)
3. Run all cells in order
4. When session expires, re-open and run all cells — training auto-resumes


In [ ]:
# Cell 1: Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    raise RuntimeError('No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# Cell 2: Authenticate & clone repo
from google.colab import userdata
import os, subprocess

TOKEN = userdata.get('GITHUB_TOKEN')
REPO = 'Sathvik-Chowdary-Veerapaneni/Language-Embeded-Agent-Action'
CLONE_URL = f'https://{TOKEN}@github.com/{REPO}.git'

if not os.path.exists('/content/leaa'):
    subprocess.run(['git', 'clone', CLONE_URL, '/content/leaa'], check=True)
else:
    subprocess.run(['git', 'pull'], cwd='/content/leaa', check=True)

subprocess.run(['git', 'config', 'user.email', 'colab@leaa.bot'], cwd='/content/leaa')
subprocess.run(['git', 'config', 'user.name', 'Colab Training Bot'], cwd='/content/leaa')
subprocess.run(['git', 'remote', 'set-url', 'origin', CLONE_URL], cwd='/content/leaa')
print('✓ Repo ready at /content/leaa')

In [ ]:
# Cell 3: Load email credentials (optional)
# Skip this cell if you don't want email notifications.
from google.colab import userdata

try:
    GMAIL_ADDRESS = userdata.get('GMAIL_ADDRESS')
    GMAIL_APP_PASSWORD = userdata.get('GMAIL_APP_PASSWORD')
    print(f'✓ Email notifications enabled → {GMAIL_ADDRESS}')
except Exception:
    GMAIL_ADDRESS = None
    GMAIL_APP_PASSWORD = None
    print('⚠ No email credentials found — notifications disabled')
    print('  Add GMAIL_ADDRESS + GMAIL_APP_PASSWORD to Colab Secrets to enable')

In [ ]:
# Cell 4: Install dependencies
%cd /content/leaa
!pip install -q -r requirements.txt
print('✓ Dependencies installed')

In [ ]:
# Cell 5: Run training
# Runs for up to 11h. Checkpoints sync to GitHub every 30 min.
# Runtime watchdog emails a warning at 10h and stops training at 11h
# so the VM has 1h to finish saving before Colab reclaims it.
# If the session expires, re-run all cells — training resumes from last checkpoint.
%cd /content/leaa
import os

cmd = 'python scripts/colab_train.py --stage 5 --timesteps 20000000 --num-envs 4 --max-runtime-hours 11'

# Append email args if credentials are available
if 'GMAIL_ADDRESS' in dir() and GMAIL_ADDRESS:
    cmd += f' --gmail {GMAIL_ADDRESS} --gmail-password {GMAIL_APP_PASSWORD}'

print(f'Running: {cmd}')
os.system(cmd)

In [ ]:
# Cell 6: (Optional) Evaluate this stage after training
%cd /content/leaa
!python rl_training/evaluate.py \\
    --model rl_training/checkpoints/wind_best.zip \\
    --vecnorm rl_training/checkpoints/vecnormalize_wind_best.pkl \\
    --stage wind \\
    --episodes 200